# Programmable Photonic Transformations

Can a programmable 2×2 photonic circuit implement a target complex optical transformation, and how do component-level imperfections affect its accuracy and transmission?

## Optical Linear Transformations

Under the  `10_directional_couplers_and_coupled_mode_theory`, `11_mach_zehnder_interferometers`

### Optical fields as complex vectors

Each component represents a power-normalized complex modal amplitude, such that its squared magnitude corresponds to the optical power carried by the associated mode.
$$\mathbf{a}_\text{in} = \begin{bmatrix}
a_1 \\
a_2
\end{bmatrix} \quad \mathbf{a}_\text{out} = \begin{bmatrix}
b_1 \\
b_2
\end{bmatrix}$$

$$P_\text{in} = \mathbf{a}_\text{in}^\dagger \mathbf{a}_\text{in}$$

$$\mathbf{a}_\text{out} = U\mathbf{a}_\text{in}$$

lossless
$$U^\dagger U = I$$

so,
$$P_\text{out} = \mathbf{a}_\text{in}^\dagger U^\dagger U \mathbf{a}_\text{in} = P_\text{in}$$

Therefore, 
$$\Vert \mathbf{a}_\text{out} \Vert^2 = \Vert \mathbf{a}_\text{in} \Vert^2$$

$$C(\theta) = \begin{bmatrix}
\cos\theta & -i\sin\theta \\
-i\sin\theta & \cos\theta
\end{bmatrix}$$

$$P(\phi) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\phi}
\end{bmatrix}$$

Ideally, $50:50$ coupler MZI is
$$U_\text{MZI}(\theta_1, \theta_2, \phi) = C(\theta_2)P(\phi)C(\theta_1) $$
$\theta_1 = \theta_2 = \frac{\pi}{4}$

An MZI that adjusts only a single internal relative phase can describe only a limited set of matrices. However, a $2 \times 2$ unitary matrix generally possesses four real degrees of freedom. If we disregard the global phase, there are three physically distinct degrees of freedom. Given this limitation, let us introduce additional phase shifters—beyond the internal phase—to control the relative phases at the input and output ports, thereby expanding the range of implementable transformations.

Let's define additional phase shifters:
$$D_\text{in}(\beta) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\beta}
\end{bmatrix}$$

$$D_\text{out}(\alpha) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\alpha}
\end{bmatrix}$$

By adding input and output phase shifters to an ideal 50:50 MZI, one can represent any $2\times2$ unitary matrix, up to a global phase. In this note, Ignore the global phase of the independent 2-port transformation.
ignore
then, we can get
$$U(\alpha, \phi, \beta) = D_\text{out}(\alpha)U_\text{MZI}(\phi)D_\text{in}(\beta)$$

when $U_t$ is unitary,
$$F_\text{shape} = \frac{|\mathrm{Tr}(U_t^\dagger U)|^2}{2\,\mathrm{Tr}(U^\dagger U)}$$

$$T_\text{avg} = \frac{\mathrm{Tr}(U^\dagger U)}{2}$$

and also,
$$U(\mathbf{p}) \approx e^{i\chi}U_t$$
$$\mathbf{p} = (\alpha, \phi, \beta)$$
This means that by varying controllable phases, a target matrix can be implemented—up to a global phase.

We need these indicators because optical losses can occur, even while the target complex matrix form is perfectly preserved.

$$U = \sqrt{\eta}\,e^{i\chi}U_t$$

$$F_\text{shape} = 1, \quad T_\text{avg} = \eta$$

### Model assumptions
- Linear, coherent, power-normalised two-mode system

- Ideal 50:50 directional couplers

- Independently controllable relative phase shifts

- No loss in the baseline unitary model

---

In [1]:
from src.verification import absolute_error, complex_error, relative_error, find_convergence_threshold
from src.waveguide import create_uniform_grid, slab_index_profile, second_derivative_matrix, slab_waveguide_operator, solve_eigenmodes, select_guided_modes
from src.coupler import double_slab_cell_average
from src.mzi import mzi_matrix, phase_matrix, coupler_matrix

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import math


COLORS = {
    "numerical": "#1F5A94",
    "numerical_alt": "#4C78A8",
    
    "analytical": "#D97706",
    "approximation": "#2E7D32",
    
    "error": "#7E57C2",
    "error_alt": "#B07AA1",

    "reference": "#6B7280",
    "reference_alt": "#9CA3AF"
}

figure_dir = Path("../figures/12_programmable_photonic_transformations")
figure_dir.mkdir(parents=True, exist_ok=True)


## Case 1 — Target Unitary Synthesis

Can a programmable $2 \times 2$ photonic circuit reproduce a prescribed complex unitary transformation using independently controllable phase shifts?

임의의 입력 복소 진폭 벡터에 대해 목표 행렬과 동일한 선형 변환을 구현할 수 있는가?, 목표 변환을 합성할 수 있는가?

$$U(\alpha, \phi, \beta) = D_\text{out}(\alpha)U_\text{MZI}(\phi)D_\text{in}(\beta)$$

$$D_\text{in}(\beta) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\beta}
\end{bmatrix}$$

$$D_\text{out}(\alpha) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\alpha}
\end{bmatrix}$$

control variant
$$\mathbf{p} = (\alpha, \phi, \beta)$$


$$U(\alpha,\phi,\beta) = D_{\mathrm{out}}(\alpha)U_{\mathrm{MZI}}(\phi)D_{\mathrm{in}}(\beta)$$

$$U(\alpha,\phi,\beta) = \begin{bmatrix}
1 & 0\\0 & e^{i\alpha}
\end{bmatrix}\frac{1}{2}\begin{bmatrix}
1-e^{i\phi} & -i(1+e^{i\phi})\\
-i(1+e^{i\phi}) & e^{i\phi}-1
\end{bmatrix}\begin{bmatrix}
1 & 0\\
0 & e^{i\beta}
\end{bmatrix}
$$

$$U(\alpha,\phi,\beta) = \frac{1}{2}\begin{bmatrix}
1-e^{i\phi} & -i(1+e^{i\phi})e^{i\beta}\\
-i(1+e^{i\phi})e^{i\alpha} & (e^{i\phi}-1)e^{i(\alpha+\beta)}
\end{bmatrix}
$$

Using Euler's formula,

$$1-e^{i\phi}=-2ie^{i\phi/2}\sin\frac{\phi}{2}$$

$$1+e^{i\phi}=2e^{i\phi/2}\cos\frac{\phi}{2}$$

Therefore,

$$U(\alpha,\phi,\beta)=e^{i(\phi/2-\pi/2)}\begin{bmatrix}
\sin\frac{\phi}{2} & e^{i\beta}\cos\frac{\phi}{2}\\
e^{i\alpha}\cos\frac{\phi}{2} & -e^{i(\alpha+\beta)}\sin\frac{\phi}{2}
\end{bmatrix}
$$

$$U^\dagger (\alpha,\phi,\beta) = e^{-i(\phi/2-\pi/2)}\begin{bmatrix}
\sin\frac{\phi}{2} & e^{-i\alpha}\cos\frac{\phi}{2}\\
e^{-i\beta}\cos\frac{\phi}{2} & -e^{-i(\alpha+\beta)}\sin\frac{\phi}{2}
\end{bmatrix}$$

$$U^\dagger U = \begin{bmatrix}
\sin^2\frac{\phi}{2} + \cos^2\frac{\phi}{2} & e^{i\beta}(\sin\frac{\phi}{2}\cos\frac{\phi}{2} - \sin\frac{\phi}{2}\cos\frac{\phi}{2}) \\
e^{-i\beta}(\sin\frac{\phi}{2}\cos\frac{\phi}{2} - \sin\frac{\phi}{2}\cos\frac{\phi}{2}) & \sin^2\frac{\phi}{2} + \cos^2\frac{\phi}{2}
\end{bmatrix}$$

$$U^\dagger U = \begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}$$

$$U^\dagger U = I$$

**Goal**:

$$U_t = \frac{1}{\sqrt{2}}\begin{bmatrix}
1 & e^{i\pi / 3} \\
e^{i\pi / 4} & -e^{i7\pi / 12}
\end{bmatrix}$$

$$
U_t^{\dagger} = \frac{1}{\sqrt{2}}\begin{bmatrix}
1 & e^{-i\pi / 4} \\
e^{-i\pi / 3} & -e^{-i7\pi / 12}
\end{bmatrix}
$$

$$U_t^{\dagger}U_t = \frac{1}{2}\begin{bmatrix}
1+1 & 0 \\
0 & 1+1
\end{bmatrix}$$

$$U_t^{\dagger}U_t = \begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$U_t^{\dagger}U_t = I$$

---

$$U(\alpha,\phi,\beta)=e^{i(\phi/2-\pi/2)}\begin{bmatrix}
\sin\frac{\phi}{2} & e^{i\beta}\cos\frac{\phi}{2}\\
e^{i\alpha}\cos\frac{\phi}{2} & -e^{i(\alpha+\beta)}\sin\frac{\phi}{2}
\end{bmatrix}
$$

$$U(\alpha,\phi,\beta)= e^{-i\pi/4}U_t$$

$$\alpha = \frac{\pi}{4}, \quad \beta = \frac{\pi}{3}, \quad \phi = \frac{\pi}{2}$$

In [2]:
def programmable_matrix(alpha, phi, beta):
    theta = np.pi / 4

    D_in = phase_matrix(beta)
    D_out = phase_matrix(alpha)
    U_mzi = mzi_matrix(theta, theta, phi)

    return D_out @ U_mzi @ D_in


In [22]:
alpha = np.pi / 4
phi = np.pi / 2
beta = np.pi / 3

U = programmable_matrix(alpha, phi, beta)

# Prescribed target
U_target = np.array([
    [1, np.exp(1j * np.pi / 3)],
    [np.exp(1j * np.pi / 4), -np.exp(1j * 7 * np.pi / 12)]
], dtype=complex) / np.sqrt(2)

print("Implemented U:")
print(U)

print("\nTarget U:")
print(U_target)

Implemented U:
[[ 0.5       -0.5j         0.6830127 +0.1830127j ]
 [ 0.70710678+0.j         -0.35355339-0.61237244j]]

Target U:
[[0.70710678+0.j         0.35355339+0.61237244j]
 [0.5       +0.5j        0.1830127 -0.6830127j ]]


In [4]:
I = np.eye(2, dtype=complex)

target_residual = np.linalg.norm(U_target.conj().T @ U_target - I, ord="fro")

unitarity_residual = np.linalg.norm(U.conj().T @ U - I, ord="fro")

overlap = np.trace(U_target.conj().T @ U)
power_trace = np.trace(U.conj().T @ U).real

F_shape = np.abs(overlap)**2 / (2 * power_trace)
T_avg = power_trace / 2

print(f"Target unitarity residual: {target_residual:.6e}")
print(f"Circuit unitarity residual: {unitarity_residual:.6e}")
print(f"Shape fidelity: {F_shape:.12f}")
print(f"Average transmission: {T_avg:.12f}")

Target unitarity residual: 1.766354e-16
Circuit unitarity residual: 6.280370e-16
Shape fidelity: 1.000000000000
Average transmission: 1.000000000000


In [5]:
chi_est = np.angle(overlap)

U_aligned = np.exp(-1j * chi_est) * U

aligned_error = ( np.linalg.norm(U_aligned - U_target, ord="fro") / np.linalg.norm(U_target, ord="fro"))

print(f"Estimated global phase: {chi_est:.12f} rad")
print(f"Phase-aligned matrix error: {aligned_error:.6e}")

assert unitarity_residual < 1e-12
assert abs(F_shape - 1.0) < 1e-12
assert abs(T_avg - 1.0) < 1e-12
assert aligned_error < 1e-12

Estimated global phase: -0.785398163397 rad
Phase-aligned matrix error: 3.223032e-16


$$U = e^{i\chi}\begin{bmatrix}
s & ce^{i\beta} \\
ce^{i\alpha} & -se^{i(\alpha + \beta)}
\end{bmatrix}$$

$$s = \sin\frac{\phi}{2}, \quad c = \cos\frac{\phi}{2}$$

$$\alpha = \arg\left(\frac{u_{21}}{u_{11}}\right), \quad \beta = \arg\left(\frac{u_{12}}{u_{11}}\right), \quad \phi = 2\text{atan}2\left(\|u_{11}\|,\|u_{12}\|\right) $$

In [6]:
def synthesize_phases(U_target):

    alpha = np.angle(U_target[1, 0] / U_target[0, 0])
    beta = np.angle(U_target[0, 1] / U_target[0, 0])

    phi = 2 * np.arctan2(np.abs(U_target[0, 0]),np.abs(U_target[0, 1]))

    return alpha, phi, beta

In [7]:
alpha_fit, phi_fit, beta_fit = synthesize_phases(U_target)

print("alpha:", alpha_fit, "rad")
print("phi:", phi_fit, "rad")
print("beta:", beta_fit, "rad")

U_reconstructed = programmable_matrix(alpha_fit, phi_fit, beta_fit)

overlap = np.trace(U_target.conj().T @ U_reconstructed)
chi = np.angle(overlap)

error = np.linalg.norm(np.exp(-1j * chi) * U_reconstructed - U_target, ord="fro")

print("Reconstruction error:", error)


alpha: 0.7853981633974483 rad
phi: 1.5707963267948966 rad
beta: 1.0471975511965976 rad
Reconstruction error: 4.558056183912393e-16


In [8]:
def synthesize_phases_new(U_target, tol=1e-12):

    U_target = np.asarray(U_target, dtype=complex)

    if U_target.shape != (2, 2):
        raise ValueError("Target must be a 2x2 matrix.")

    if not np.allclose(U_target.conj().T @ U_target, np.eye(2), atol=tol, rtol=0):
        raise ValueError("Target must be unitary.")

    # Off-diagonal ft
    if np.abs(U_target[0, 0]) < tol:
        
        alpha = np.angle(U_target[1, 0] / U_target[0, 1])
        beta = 0.0

        phi = 0.0

    # Diagonal ft
    elif np.abs(U_target[0, 1]) < tol:
        
        alpha = np.angle(-U_target[1, 1] / U_target[0, 0])
        beta = 0.0

        phi = np.pi

    else:
        
        alpha = np.angle(U_target[1, 0] / U_target[0, 0])
        beta = np.angle(U_target[0, 1] / U_target[0, 0])

        phi = 2 * np.arctan2(np.abs(U_target[0, 0]), np.abs(U_target[0, 1]) )

    return alpha, phi, beta

In [9]:
targets = {
    "Identity": np.eye(2, dtype=complex),
    "SWAP": np.array([[0, 1], [1, 0]], dtype=complex),
    "Complex Unitary": U_target
}

for name, target in targets.items():
    alpha, phi, beta = synthesize_phases_new(target)
    U = programmable_matrix(alpha, phi, beta)

    overlap = np.trace(target.conj().T @ U)
    chi = np.angle(overlap)

    error = (np.linalg.norm(np.exp(-1j * chi) * U - target,ord="fro") / np.linalg.norm(target, ord="fro"))

    print(f"{name}: reconstruction error = {error:.6e}")

Identity: reconstruction error = 2.383330e-16
SWAP: reconstruction error = 2.303328e-16
Complex Unitary: reconstruction error = 3.223032e-16


### Results

- Identity: 입력된 모드가 서로 혼합되지 않고 유지된다.

- Swap: 입력된 모드가 서로 교환된다.

- Complex Unitary: 광전력의 혼합 비율을 제어하는 것을 넘어 출력 복소 진폭의 상대 위상까지 제어할 수 있다.

#### Limitations
이상적 $2 \times 2$ programmable MZI의 축약된 행렬 모델에 대한 수치 검증에 그친다.

---
## Case 2 — Coupling Errors and Reachability

When directional couplers deviate from ideal $50:50$ splitting, which target unitary transformations remain achievable?

제작 오차가 그 구현 가능성을 어떻게, 얼마나 제한하는가?

$$ U_\text{non-ideal MZI} (\theta_1, \theta_2, \phi) = C(\theta_2)P(\phi)C(\theta_1)$$
$$U_{11} = \cos\theta_1\cos\theta_2 - \sin\theta_1\sin\theta_2 e^{i\phi}$$

It $0 \le \theta_1, \theta_2 \le \frac{\pi}{2}$,
$$|\cos(\theta_1 + \theta_2)| \le |U_{11}| \le |\cos(\theta_1 - \theta_2)|$$

Furthermore,
$$|U_\text{11}|^2 = (\cos\theta_1\cos\theta_2 - \sin\theta_1\sin\theta_2 e^{i\phi}) \times (\cos\theta_1\cos\theta_2 - \sin\theta_1\sin\theta_2 e^{-i\phi})$$

Because of $-1 \le \cos\phi \le 1$,
$$\min_\phi |U_\text{11}|^2 = \cos^2(\theta_1 + \theta_2), \quad \max_\phi |U_\text{11}|^2 = \cos^2(\theta_1 - \theta_2)$$

Therefore, 
$$\cos^2(\theta_1 + \theta_2) \le |\,U_{11}|^2 \le \cos^2(\theta_1 - \theta_2)$$

**Initial condition**

$$g_0 = 0.5 \mathrm{\mu m}$$
$$\kappa_0 = 0.088084729155 \mathrm{\mu m}^{-1} $$
$$\gamma = 2.738058478477 \mathrm{\mu m}^{-1}$$
$$L_\text{DC} = \frac{\pi}{4\kappa_0}$$
$$\text{Gap perturbation} = \pm \,0.05\mathrm{\mu m}$$

$\text{Ideal}: g_1 = g_2 = g_0$

$\text{Common-mode error}: g_1 = g_2 = g_0 + 0.05$

$\text{Differential-mode error}: g_1 = g_0 + 0.05, \,\, g_2 = g_0 - 0.05$

In [10]:
g0 = 0.5
kappa0 = 0.088084729155
gamma = 2.738058478476613

L_DC = np.pi / (4 * kappa0)

In [11]:
def theta_from_gap(g):
    kappa = kappa0 * np.exp(-gamma * (g - g0))
    return kappa * L_DC

In [12]:
gap_cases = {
    "Ideal": (g0, g0),
    "Common-mode": (g0 + 0.05, g0 + 0.05),
    "Differential-mode": (g0 + 0.05, g0 - 0.05)
}

phi_values = np.linspace(0, 2 * np.pi, 721)

for name, (g1, g2) in gap_cases.items():

    theta1 = theta_from_gap(g1)
    theta2 = theta_from_gap(g2)

    P_analytical_min = np.cos(theta1 + theta2)**2
    P_analytical_max = np.cos(theta1 - theta2)**2

    P_numerical = np.array([
        np.abs(mzi_matrix(theta1, theta2, phi)[0, 0])**2
        for phi in phi_values
    ])

    P_numerical_min = np.min(P_numerical)
    P_numerical_max = np.max(P_numerical)

    error_min = abs(P_numerical_min - P_analytical_min)
    error_max = abs(P_numerical_max - P_analytical_max)

    print(f"\n{name}")
    print(f"Analytical range: [{P_analytical_min:.8f}, {P_analytical_max:.8f}]")
    print(f"Numerical range:  [{P_numerical_min:.8f}, {P_numerical_max:.8f}]")
    print(f"Bound errors: {error_min:.3e}, {error_max:.3e}")

    assert max(error_min, error_max) < 1e-12



Ideal
Analytical range: [0.00000000, 1.00000000]
Numerical range:  [0.00000000, 1.00000000]
Bound errors: 2.344e-32, 0.000e+00

Common-mode
Analytical range: [0.03985035, 1.00000000]
Numerical range:  [0.03985035, 1.00000000]
Bound errors: 6.939e-18, 2.220e-16

Differential-mode
Analytical range: [0.00021735, 0.95418268]
Numerical range:  [0.00021735, 0.95418268]
Bound errors: 3.388e-18, 0.000e+00


common-mode는 identity transformation을 구현 가능,

완전한 SWAP transformation은 구현할 수 없고, 최소 3.985%의 해당 포트 출력 전력이 잔존

Differential-mode의 경우 완전한 Identity와 SWAP 모두 구현할 수 없다.

SWAP의 잔존 출력 전력은 최소 약 0.0217%. Identity의 경우 원하는 포트로 전달할 수 없는 전력이 약 4.582%

In [13]:
tol = 1e-12

for name, (g1, g2) in gap_cases.items():
    theta1 = theta_from_gap(g1)
    theta2 = theta_from_gap(g2)

    P_min = np.cos(theta1 + theta2)**2
    P_max = np.cos(theta1 - theta2)**2

    print(f"\n{name}")

    for target_name, U_target in targets.items():
        P_target = np.abs(U_target[0, 0])**2

        reachable = ( P_min - tol <= P_target <= P_max + tol)

        print(f"{target_name}: {reachable}")


Ideal
Identity: True
SWAP: True
Complex Unitary: True

Common-mode
Identity: True
SWAP: False
Complex Unitary: True

Differential-mode
Identity: False
SWAP: False
Complex Unitary: True


If a target unitary transformation is unreachable, what is the highest shape fidelity achievable through phase optimization?

$$p_t = |(U_t)_{11}|^2 $$

$$p = |U_{11}|^2$$

두 행렬이 unitary이고, 상대위상을 최적으로 조절할 수 있을 때, 코시-슈바르츠 부등식에 의해
$$\text{Tr}(U_t^\dagger U) = \sqrt{p_tp}e^{i\phi_1} + \sqrt{(1-p_t)(1-p)}e^{i\phi_2}$$

$$|\text{Tr}(U_t^\dagger U)| = |\sqrt{p_tp}e^{i\phi_1} + \sqrt{(1-p_t)(1-p)}e^{\phi_2} |\le 2\left[\sqrt{p_tp} + \sqrt{(1-p_t)(1-p)}\right]$$

상대위상 $\phi_1$, $\phi_2$에 대해,
$$\phi_1 = \phi_2 \,\,(\text{mode}\,\, 2\pi)$$ 일때,

$$F_\text{shape, max}(p) = \left[ \sqrt{p_tp} + \sqrt{(1-p_t)(1-p)} \right]^2$$

1에 가까우면 거의 동일. 0이면 직교상태, 완전히 다름

In [14]:
def maximum_shape_fidelity(p_target, p_min, p_max):
    p_best = np.clip(p_target, p_min, p_max)

    F_max = (np.sqrt(p_target * p_best) + np.sqrt((1 - p_target) * (1 - p_best)))**2

    return F_max

In [15]:
for name, (g1, g2) in gap_cases.items():
    theta1 = theta_from_gap(g1)
    theta2 = theta_from_gap(g2)

    P_min = np.cos(theta1 + theta2)**2
    P_max = np.cos(theta1 - theta2)**2

    print(f"\n{name}")

    for target_name, target in targets.items():
        P_target = np.abs(target[0, 0])**2

        F_max = maximum_shape_fidelity(P_target, P_min, P_max)

        print(f"{target_name}: F_max = {F_max:.8f}")



Ideal
Identity: F_max = 1.00000000
SWAP: F_max = 1.00000000
Complex Unitary: F_max = 1.00000000

Common-mode
Identity: F_max = 1.00000000
SWAP: F_max = 0.96014965
Complex Unitary: F_max = 1.00000000

Differential-mode
Identity: F_max = 0.95418268
SWAP: F_max = 0.99978265
Complex Unitary: F_max = 1.00000000


In [16]:
for name, (g1, g2) in gap_cases.items():

    theta1 = theta_from_gap(g1)
    theta2 = theta_from_gap(g2)

    p_min = np.cos(theta1 + theta2)**2
    p_max = np.cos(theta1 - theta2)**2

    print(f"\n{name}")

    for target_name, target in targets.items():

        p_target = np.abs(target[0, 0])**2
        p_best = np.clip(p_target, p_min, p_max)

        A = np.cos(theta1) * np.cos(theta2)
        B = np.sin(theta1) * np.sin(theta2)

        cos_phi = (A**2 + B**2 - p_best) / (2 * A * B)
        phi = np.arccos(np.clip(cos_phi, -1, 1))

        M = mzi_matrix(theta1, theta2, phi)

        beta = 0.0

        if p_target < 1e-12:
            # SWAP target
            alpha = (np.angle(target[1, 0] / target[0, 1]) - np.angle(M[1, 0] / M[0, 1]))

        elif 1 - p_target < 1e-12:
            # Identity target
            alpha = (np.angle(target[1, 1] / target[0, 0]) - np.angle(M[1, 1] / M[0, 0]))

        else:
            # General complex unitary target
            alpha = (np.angle(target[1, 0] / target[0, 0])- np.angle(M[1, 0] / M[0, 0]))
            beta = (np.angle(target[0, 1] / target[0, 0]) - np.angle(M[0, 1] / M[0, 0]))

        U = phase_matrix(alpha) @ M @ phase_matrix(beta)

        overlap = np.trace(target.conj().T @ U)
        power_trace = np.trace(U.conj().T @ U).real

        F_numerical = np.abs(overlap)**2 / (2 * power_trace)
        F_analytical = maximum_shape_fidelity(p_target, p_min, p_max)

        error = abs(F_numerical - F_analytical)

        print(
            f"{target_name}: "
            f"F = {F_numerical:.8f}, "
            f"error = {error:.3e}"
        )

        assert error < 1e-12



Ideal
Identity: F = 1.00000000, error = 0.000e+00
SWAP: F = 1.00000000, error = 0.000e+00
Complex Unitary: F = 1.00000000, error = 0.000e+00

Common-mode
Identity: F = 1.00000000, error = 0.000e+00
SWAP: F = 0.96014965, error = 0.000e+00
Complex Unitary: F = 1.00000000, error = 2.220e-16

Differential-mode
Identity: F = 0.95418268, error = 1.110e-16
SWAP: F = 0.99978265, error = 1.110e-16
Complex Unitary: F = 1.00000000, error = 2.220e-16


Lossless가 Universally reacable을 의미하는 건 아님.

같은 coupler Differenctial mode 조건아래에서도 두 목표 행렬의 최대 fidelity가 꽤나 다르다.

$$F^\text{Identity}_\text{max} = 0.95418268$$
$$F^\text{SWAP}_\text{max} = 0.99978265$$

소자 성능을 평가할 때 단순한 coupling error의 크기 뿐만 아닐 어떤 광학적 변환을 구현하고자 했는지도 중요하다는 의미

추가로 Phase compensation에는 구조적 한계가 존재. 
목표하는 혼합비가 reachable intervarl안에 있다면 Phase 제어를 통해 F = 1을 달성할 수 있으나, initerval 밖에 있다면 $\alpha, \phi, \beta$ 를 아무리 조절해도 fidelit 상한이 존재하게 됨

### Results and Discussion (written after all numerical studies)

`10_directional_couplers_and_coupled_mode_theory` 와 `11_mach_zehnder_interferometers`에서 도출 및 가정한 parameter들을 그대로 이어 사용하였다. 

$|U_\text{11}|^2$ range를 통해 구현 가능한 transformation이 mode에 따라 다름을 확인하였다. leakage와 연결되는 복소 행렬의 형태는 circuit의 performance를 평가하는 중요한 요소 중 하나인데, 동일한 평균 광출력이더라도 복소 행렬의 완전성을 보장할 수 없음을 보이기 위해 case 3에서 fidelity를 transmission과 구분하여 활용하였다.

## Case 3 — Optical Loss and Fidelity

Can a photonic circuit preserve the target transformation's shape fidelity while suffering optical transmission loss?

$$U_\text{ideal} = D_\text{out}(\alpha)CP(\phi)CD_\text{in}(\beta)$$

Loss $L$,
$$L = \begin{bmatrix}
\sqrt{\eta_1} & 0 \\
0 & \sqrt{\eta_2}
\end{bmatrix}$$

$\eta_1$, $\eta_2$ is power transmission of each arm,
$$0 \le \eta_1, \eta_2 \le 1$$

$$U_\text{loss} = D_\text{out}(\alpha)CP(\phi)LCD_\text{in}(\beta)$$

---

$$F_\text{shape} = \frac{|\text{Tr}(U_t^\dagger U_\text{loss}) |^2}{2\text{Tr}(U_\text{loss}^\dagger U_\text{loss})}$$

$$T_\text{avg} = \frac{\text{Tr}(U_\text{loss}^\dagger U_\text{loss})}{2}$$



**Uniform Loss**

$$\eta_1 = \eta_2 = \eta$$

then,
$$L = \sqrt{\eta}I$$


$$U_\text{loss} = \sqrt{\eta}\,U_\text{ideal}$$

Therefore, $\eta > 0$,
$$F_\text{shape} = 1, \quad T_\text{avg} = \eta$$:
광전력은 감소하지만 복소 행렬의 상대적인 형태는 유지된다. 

Although the optical power decreases, the relative form of the complex matrix is preserved.

**Differential Loss**

$$\eta_1 \neq \eta_2$$

When utilizing the trace properties of unitary matrices,
$$\text{Tr}(U_t^\dagger U_\text{loss}) = e^{i\chi}(\sqrt{\eta_1} + \sqrt{\eta_2})$$

$\chi$: global phase between the ideal implementation matrix and the target matrix

$$\text{Tr}(U_\text{loss}^\dagger U_\text{loss}) = \eta_1 + \eta_2$$

Therefor,
$$F_\text{shape} = \frac{(\sqrt{\eta_1} + \sqrt{\eta_2})^2}{2(\eta_1 + \eta_2)}$$
$$T_\text{avg} = \frac{\eta_1 + \eta_2}{2}$$

In [17]:
def lossy_programmable_matrix(alpha, phi, beta, eta1, eta2):

    C = coupler_matrix(np.pi / 4)

    L = np.diag([np.sqrt(eta1), np.sqrt(eta2)])

    U = (phase_matrix(alpha) @ C @ phase_matrix(phi) @ L @ C @ phase_matrix(beta))

    return U

In [18]:
alpha = np.pi / 4
phi = np.pi / 2
beta = np.pi / 3

loss_cases = {
    "Uniform": (0.65, 0.65),
    "Differential": (0.90, 0.40)}

In [19]:
for name, (eta1, eta2) in loss_cases.items():

    U = lossy_programmable_matrix(alpha, phi, beta, eta1, eta2)

    trace_power = np.trace(U.conj().T @ U).real

    F_numerical = (np.abs(np.trace(U_target.conj().T @ U))**2 / (2 * trace_power))

    T_numerical = trace_power / 2

    F_analytical = ((np.sqrt(eta1) + np.sqrt(eta2))**2 / (2 * (eta1 + eta2)))

    T_analytical = (eta1 + eta2) / 2

    print(f"\n{name}")
    print(f"Numerical fidelity: {F_numerical:.8f}")
    print(f"Analytical fidelity: {F_analytical:.8f}")
    print(f"Average transmission: {T_numerical:.8f}")

    assert abs(F_numerical - F_analytical) < 1e-12
    assert abs(T_numerical - T_analytical) < 1e-12


Uniform
Numerical fidelity: 1.00000000
Analytical fidelity: 1.00000000
Average transmission: 0.65000000

Differential
Numerical fidelity: 0.96153846
Analytical fidelity: 0.96153846
Average transmission: 0.65000000


평균 전송 효율이 같아도 transformation fidelity는 달라질 수 있다.

In [20]:
a_single = np.array([1, 0], dtype=complex)

a_plus = np.array([1, 1j * np.exp(-1j * beta)], dtype=complex) / np.sqrt(2)

a_minus = np.array([ 1, -1j * np.exp(-1j * beta)], dtype=complex) / np.sqrt(2)

inputs = {
    "Single-port": a_single,
    "Coherent (+)": a_plus,
    "Coherent (-)": a_minus}

loss_cases = {
    "Uniform": (0.65, 0.65),
    "Differential": (0.90, 0.40)}


In [21]:
for name, (eta1, eta2) in loss_cases.items():

    U = lossy_programmable_matrix(alpha, phi, beta, eta1, eta2)

    print(f"\n{name}")

    for state_name, a_in in inputs.items():

        a_out = U @ a_in

        P_in = np.vdot(a_in, a_in).real
        P_out = np.vdot(a_out, a_out).real

        T = P_out / P_in

        print(f"{state_name}: T = {T:.8f}")


Uniform
Single-port: T = 0.65000000
Coherent (+): T = 0.65000000
Coherent (-): T = 0.65000000

Differential
Single-port: T = 0.65000000
Coherent (+): T = 0.90000000
Coherent (-): T = 0.40000000


**Uniform loss**

$$L = \sqrt{0.65}I$$
어떤 normalized 된 입력을 사용해도 0.65

**Differential loss**

$$L = \begin{bmatrix}
\sqrt{0.9} & 0 \\
0 & \sqrt{0.4} 
\end{bmatrix}$$

입력 복소 진폭의 상대 위상에 따라 두 arm의 광전력 분배에 차이가 생긴다. 손실이 서로 다른 경로에 빛을 어떻게 분배하느냐에 따라 전체 전송 효율이 변한다. 

$$\mathbf{a_+}; \quad T = 0.9$$
$$\mathbf{a_-}; \quad T = 0.4$$

$$T(\mathbf{a}) = \frac{\mathbf{a}^\dagger U^\dagger U\mathbf{a}}{\mathbf{a}^\dagger\mathbf{a}}$$

$$U^\dagger U = D_\text{in}^\dagger C^\dagger \begin{bmatrix}
\eta_1 & 0 \\
0 & \eta_2 
\end{bmatrix}CD_\text{in}$$

$U^\dagger U $의 eigenvalue는
$$\lambda_1 = \eta_1, \quad \lambda_2 = \eta_2$$

$$\min(\eta_1, \eta_2) \le T(\mathbf{a}) \le \max(\eta_1, \eta_2)$$

$\mathbf{a_+}$, $\mathbf{a_-}$는 현재 모델에서 가능한 transmission 전체 범위를 검증한 것.

---

### Results and Discussion (written after all numerical studies)

Uniform attenuation는 광전송 효율은 감소하지만, 목표한 복소 변환의 shape fidelity는 유지된다. Differential attenuation의 경우 평균 전송 효율이 같더라도 상대 진폭이 왜곡되어 shape fidelity가 감소할 수 있다. Input-state dependence에서는 Differential loss에서는 입력의 복소 진폭 및 상대 위상에 따라 전송 효율이 달라진다. 이를 통해 Photonic circuit 성능을 평균 transmission 하나로 평가하는 것은 충분하지 않다는 점을 알 수 있으며, 달성하고자 하는 혼합비의 값이 $|U_\text{11}|^2$ range 안에 없으면 $\alpha$, $\beta$, $\phi$를 조절하여도 달성 가능한 fidelity의 상한이 존재하게 된다. 

이는 $2 \times 2$ reduced-order matrix model에 기반한 결과이며, 실제 3D 광소자의 손실을 검증한 것이 아니다. 손실은 두 개의 MZI are의 감쇠 계수로 모델링 하였으며, 반사나 살란에 따른 추가적인 모드 결합은 고려하지 않았다. 현재의 shape fidelity는 전체 변환 행렬의 유사도를 평가하는 지표이며, 개별 입력 상태의 fidelity와 동일한 개념을 사용한 것은 아니다. 